# Prompt Chaining con Tools + Reasoning

Mismo patrón secuencial que el notebook anterior (vuelos→alojamiento→actividades→transporte→itinerario), pero ahora los agentes tienen acceso a herramientas de búsqueda web y geolocalización para obtener datos reales.

Diferencia con la versión basic: aquí cada agente puede buscar información en internet y calcular distancias reales en vez de inventarlas.

In [1]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## Qué son las Tools

Clases Python que el agente puede invocar durante su ejecución. Se pasan como lista en `tools=[...]`.

### SerperDevTool

Búsquedas en Google via la API de Serper. Necesita `SERPER_API_KEY` en el `.env`.

| Parámetro | Qué hace |
|-----------|----------|
| `country` | País para localizar resultados (p.ej. `"ES"`). |
| `locale` | Idioma de los resultados (p.ej. `"es"`). |
| `location` | Ubicación geográfica para el ranking. |
| `tbs` | Filtro temporal de Google (`qdr:y2` = últimos 2 años). |
| `n_results` | Cantidad de resultados a devolver. |

### google_maps_distance (custom tool)

Tool propia que calcula distancias y tiempos reales entre dos puntos usando la Distance Matrix API de Google. Acepta nombres de lugares, direcciones o coordenadas (hace geocoding interno si le pasas un nombre ambiguo como "Cascada Skógafoss"). Necesita `GOOGLE_MAPS_API_KEY` en el `.env`.

| Parámetro | Qué hace |
|-----------|----------|
| `origin` | Punto de partida (nombre, dirección o coordenadas). |
| `destination` | Punto de llegada. |
| `mode` | Medio de transporte: `driving`, `transit`, `walking`, `bicycling`. |

### `reasoning=True`

Activa un paso de razonamiento interno antes de que el agente responda. El agente planifica su enfoque antes de ejecutar, lo que mejora la calidad en tareas complejas como ensamblar un itinerario completo.

## La Crew con tools (`viajes_crew.py`)

Misma estructura que la versión basic, pero cada agente tiene herramientas asignadas:

- `vuelos`, `alojamiento`, `actividades`: usan `SerperDevTool` para buscar en Google
- `transporte`, `coche`: usan `SerperDevTool` + `google_maps_distance` para buscar opciones y calcular distancias/tiempos reales
- `itinerario`: usa `reasoning=True` para planificar mejor el ensamblaje final

In [3]:
%pycat viajes_crew.py

from __future__ import annotations

from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from tools import google_maps_distance

tool_search = SerperDevTool(
    country="ES",
    locale="es",
    location="Barcelona, Spain",
    tbs="qdr:y2",
    n_results=10
)

@CrewBase
class ViajesCrew:
    """Crew con tools de búsqueda y mcps asignadas a los agentes."""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def vuelos(self) -> Agent:
        return Agent(config=self.agents_config["vuelos"], tools=[SerperDevTool()])

    @agent
    def alojamiento(self) -> Agent:
        return Agent(config=self.agents_config["alojamiento"], tools=[SerperDevTool()])

    @agent
    def actividades(self) -> Agent:
        return Agent(config=self.agents_config["actividades"], tools=[SerperDevTool()])

    @agent
    def transporte(self) -> Agent:
        return Agent(confi

## Ejecución

In [4]:
from viajes_crew import ViajesCrew

inputs = {
    "destino": "Islandia",
    "dias": 5,
    "personas": 2,
    "presupuesto": 2200,
}

trip = ViajesCrew()
result = await trip.crew().kickoff_async(inputs=inputs)
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: ViajesCrew                                                                                               │
│  ID: d70dd224-4334-476e-bd82-49387edb0169                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: vuelos_task                                                                                              │
│  ID: 20add1d8-3a49-4e26-aff6-afb4989b343d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Especialista en Vuelos                                                                                  │
│                                                                                                                 │
│  Task: Propon 2-3 opciones de vuelo a Islandia para 2 personas y 5 dias. Presupuesto total del viaje: 2200      │
│  EUR.                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'vuelos baratos a Reykjavik Islandia para 2 personas 5 dias presupuesto 2200 EUR'}      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_the_internet_with_serper                                                                          │
│  Args: {'search_query': 'mejores ofertas vuelo a Islandia desde Europa 2 personas 5 dias'}                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Director de Itinerario                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  No he podido obtener las distancias y tiempos por un error con la API de Google Maps. Sin embargo, cuento con  │
│  referencias aproximadas para cada trayecto para montar el itinerario organizado y coherente con horarios y     │
│  costes.                                                                                                        │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Itinerario día a día para 2 personas en Islandia (5 días)                                                    │
│                                                                                                                 │
│  ## Día 1: Llegada y paseo por Reykjavik                                                                        │
│  - **Horario:**                                                                                                 │
│    - 13:00 Llegada a Aeropuerto Reykjavik-Keflavik                                                              │
│    - 14:00 Traslado en bus Flybus a hotel/Airbnb en centro Reykjavik (45 km, ~45 min)                           │
│    - 15:00 Check-in y descanso                                                                                  │
│    - 16:30 Paseo a pie por centro histórico (Miðborg)                                                           │
│    - 19:30 Cena en restaurante local (Fish Market o similar)                                                    │
│  - **Desplazamientos:**                                                                                         │
│    - Aeropuerto -> Reykjavik centro: 45 km, 45 min (bus)                                                        │
│    - Paseos a pie dentro centro histórico                                                                       │
│  - **Alojamiento:** Airbnb en Reykjavik centro                                                                  │
│  - **Coste día 1:**                                                                                             │
│    - Bus aeropuerto ida y vuelta: 60 EUR                                                                        │
│    - Cena restaurante: 70 EUR                                                                                   │
│    - Alojamiento (5 noches, prorrateado): 100 EUR                                                               │
│    - **Total Día 1:** 230 EUR                                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Día 2: Golden Circle                                                                                        │
│  - **Horario aproximado:**                                                                                      │
│    - 08:00 Recogida coche alquiler en Reykjavik        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: itinerario_task                                                                                          │
│  Agent: Director de Itinerario                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: ViajesCrew                                                                                               │
│  ID: d70dd224-4334-476e-bd82-49387edb0169                                                                       │
│  Final Output: No he podido obtener las distancias y tiempos por un error con la API de Google Maps. Sin        │
│  embargo, cuento con referencias aproximadas para cada trayecto para montar el itinerario organizado y          │
│  coherente con horarios y costes.                                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Itinerario día a día para 2 personas en Islandia (5 días)                                                    │
│                                                                                                                 │
│  ## Día 1: Llegada y paseo por Reykjavik                                                                        │
│  - **Horario:**                                                                                                 │
│    - 13:00 Llegada a Aeropuerto Reykjavik-Keflavik                                                              │
│    - 14:00 Traslado en bus Flybus a hotel/Airbnb en centro Reykjavik (45 km, ~45 min)                           │
│    - 15:00 Check-in y descanso                                                                                  │
│    - 16:30 Paseo a pie por centro histórico (Miðborg)                                                           │
│    - 19:30 Cena en restaurante local (Fish Market o similar)                                                    │
│  - **Desplazamientos:**                                                                                         │
│    - Aeropuerto -> Reykjavik centro: 45 km, 45 min (bus)                                                        │
│    - Paseos a pie dentro centro histórico                                                                       │
│  - **Alojamiento:** Airbnb en Reykjavik centro                                                                  │
│  - **Coste día 1:**                                                                                             │
│    - Bus aeropuerto ida y vuelta: 60 EUR                                                                        │
│    - Cena restaurante: 70 EUR                                                                                   │
│    - Alojamiento (5 noches, prorrateado): 100 EUR                                                               │
│    - **Total Día 1:** 230 EUR                                                                                   │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Día 2: Golden Circle                                                                                        │
│  - **Horario aproximado:**                                                                                      │
│    - 08:00 Recogida coche alquiler en Reykjavik       

No he podido obtener las distancias y tiempos por un error con la API de Google Maps. Sin embargo, cuento con referencias aproximadas para cada trayecto para montar el itinerario organizado y coherente con horarios y costes.

---

# Itinerario día a día para 2 personas en Islandia (5 días)

## Día 1: Llegada y paseo por Reykjavik  
- **Horario:**  
  - 13:00 Llegada a Aeropuerto Reykjavik-Keflavik  
  - 14:00 Traslado en bus Flybus a hotel/Airbnb en centro Reykjavik (45 km, ~45 min)  
  - 15:00 Check-in y descanso  
  - 16:30 Paseo a pie por centro histórico (Miðborg)  
  - 19:30 Cena en restaurante local (Fish Market o similar)  
- **Desplazamientos:**  
  - Aeropuerto -> Reykjavik centro: 45 km, 45 min (bus)  
  - Paseos a pie dentro centro histórico  
- **Alojamiento:** Airbnb en Reykjavik centro  
- **Coste día 1:**  
  - Bus aeropuerto ida y vuelta: 60 EUR  
  - Cena restaurante: 70 EUR  
  - Alojamiento (5 noches, prorrateado): 100 EUR  
  - **Total Día 1:** 230 EUR  

---

## Dí

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ dc110d9e-3b15-47b4-8aec-d0ae8f6e6198                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/trace_batches/dc110d9e-3b15-47b4-8aec-d0a │
│ e8f6e6198                                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯
